Calendar Bot

In [1]:
# Imports
from datetime import date
import calendar
import pandas as pd
import numpy as np
import re
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

/home/sofi/Documents/vscode/Archivos_DL/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Detalles de modelo
MODEL_NAME = 'Qwen/Qwen2.5-Coder-1.5B-Instruct'
MAX_STEPS = 5

In [3]:
WEEKDAY_CONVERTER = ['monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday']

MONTHS = ['january', 'february', 'march', 'april', 'may', 'june', 'july', 'august', 'september', 'october', 'november', 'december']

In [4]:
# Tooling
def get_weekday(day: int, month: int, year: int) -> str:
    d = date(year, month, day)
    return WEEKDAY_CONVERTER[d.weekday()]

def count_weekday(weekday: int, month: int, year: int) -> int:
    total_count = 0
    for i in range(1, (calendar.monthrange(year, month)[1])+1):
        if date(year, month, i).weekday() == weekday:
            total_count += 1
    
    return total_count

def days_between(day1: int, month1: int, year1: int, day2: int, month2: int, year2: int) -> int:
    return abs(((date(year1, month1, day1)) - (date(year2, month2, day2))).days)

TOOLS = {
    'get_weekday': get_weekday,
    'count_weekday': count_weekday,
    'days_between': days_between
}

In [5]:
days_between(19, 4, 2026, 1, 4, 2026)

18

In [6]:
# Cargando Modelo
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map='cuda'
)

def llm(messages, max_new_tokens=200):
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

SYSTEM_PROMPT = """
You are an agent that solves calendar related queries.

You have access to these tools:
- get_weekday(day, month, year): returns the day of the week as a string
- count_weekday(weekday, month, year): returns the amount of specific weekdays in a month in a year as an integer
- days_between(day1, month1, year1, day2, month2, year2): returns the amount of days between two dates as an integer

On each turn answer EXACTLY in one of these two formats:
1) If you need to use a tool (use only positional arguments):
ACTION: tool_name(arg_1, arg_2, arg_3)

If you find a string that contains something like 17/04/2005, the first number is the day, the second one is the month and the third one is the year, every time!
If you find a string that contains something like "April 17, 2005", turn the name of the month to an integer from 1 to 12 depending on the month, and take the day and years as integers.
When using count_weekday, weekday MUST be an integer: Monday=0, ..., Sunday=6. Do NOT output strings.

2) If you already have the final answer:
ANSWER: <answer>

Do not explain anything else. Just ONE line with ACTION or ANSWER.
"""

def parse_response(text: str):
    text = text.strip()

    m_answer = re.search(r"ANSWER:\s*(.+)", text)
    if m_answer:
        return ("answer", m_answer.group(1).strip())

    m_action = re.search(r"(?:ACTION:\s*)?(\w+)\(([^)]*)\)", text)
    if m_action:
        name = m_action.group(1)
        raw_args = [a.strip() for a in m_action.group(2).split(",") if a.strip()]
        args = []
        for a in raw_args:
            if '=' in a:
                a = a.split("=", 1)[1].strip()
            args.append(int(a))
        return ("action", (name, args))
    
    return ("unknown", text)

def run_agent(task: str, max_steps: int = MAX_STEPS):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': task}
    ]
    for step in range(1, max_steps+1):
        print(f'STEP {step}')

        response = llm(messages)
        print(f"[LLM]: {response}")
        messages.append({'role': 'assistant', 'content': response})

        kind, payload = parse_response(response)
        print(kind, payload)

        if kind == 'answer':
            print(f'\n Final Answer: {payload}')
            return payload

        elif kind == 'action':
            name, args = payload
            if name not in TOOLS:
                observation = f'Error: The tool {name} does not exist'
            else:
                try:
                    result = TOOLS[name](*args)
                    observation = f'Result of tool {name}({args}): {result}'
                except Exception as e:
                    observation = f'Error executing {name}: {e}'
            print(f'TOOL: {observation}')
            messages.append({'role': 'user', 'content': observation})
        
        else:
            print('WARNING: NO PARSING WAS POSSIBLE. BREAKING')
            break
    
    print('\nSTEP_LIMIT REACHED, no final answer found')
    return None

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 338/338 [00:00<00:00, 524.31it/s]


In [9]:
# Example
user_task = 'How many days between 18/04/2026, 01/01/1970?'
run_agent(user_task, MAX_STEPS)

STEP 1
[LLM]: days_between(18, 4, 2026, 1, 1, 1970)
action ('days_between', [18, 4, 2026, 1, 1, 1970])
TOOL: Result of tool days_between([18, 4, 2026, 1, 1, 1970]): 20561
STEP 2
[LLM]: ANSWER: 20561
answer 20561

 Final Answer: 20561


'20561'

In [10]:
# Example
user_task = 'What day of the week was April 18, 2005?'
run_agent(user_task, MAX_STEPS)

STEP 1
[LLM]: get_weekday(18, 4, 2005)
action ('get_weekday', [18, 4, 2005])
TOOL: Result of tool get_weekday([18, 4, 2005]): monday
STEP 2
[LLM]: ANSWER: monday
answer monday

 Final Answer: monday


'monday'